# RECOLECTION OF DATA. SCRAPPING YOUTUBE COMMENTS USING API.

In [ ]:
channel_ids = ['UCDYWzPdgUz5tK8SkEz_lHfg'] #David Bisbal id channel

In [ ]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from concurrent.futures import ThreadPoolExecutor
from google.colab import files

In [ ]:
import json
import subprocess
import pandas as pd
import os

In [ ]:
from oauth2client.client import GoogleCredentials

from google.colab import auth as google_auth
google_auth.authenticate_user()

from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
!pip install -U yt-dlp
!apt-get install ffmpeg

# API AUTHENTICITY

In [ ]:
# Youtube API key (https://research.youtube/)
#https://console.cloud.google.com
api_key = ' ' #<- introduce your API key

In [ ]:
api_service_name = "youtube"
api_version = "v3"

# Get credentials and create an API client
youtube = build(
    api_service_name, api_version, developerKey=api_key)

# CANAL IDENTIFICATION

In [ ]:
def get_channel_stats(youtube, channel_ids):
    all_data = []
    request = youtube.channels().list(
        part="snippet,contentDetails,statistics",
        id=','.join(channel_ids)
    )
    response = request.execute()

    # loop through items
    for item in response['items']:
      data = {'channelName': item['snippet']['title'],
              'subscribers': item['statistics']['subscriberCount'],
              'views': item['statistics']['viewCount'],
              'totalVideos': item['statistics']['videoCount'],
              'playlistId': item['contentDetails']['relatedPlaylists']['uploads']
      }

      all_data.append(data)

    return(pd.DataFrame(all_data))

In [ ]:
channel_stats = get_channel_stats(youtube, channel_ids)
channel_stats

In [ ]:
playlist_id = channel_stats.loc[0, 'playlistId']
playlist_id

# IDENTIFICATION OF CANAL CONTENT

In [ ]:
def get_video_ids(youtube, playlist_id):
    video_ids = []

    # Request the first page of results
    request = youtube.playlistItems().list(
        part="snippet, contentDetails",
        playlistId=playlist_id,
        maxResults=50
    )
    response = request.execute()

    # Extract video IDs from the first page
    for item in response['items']:
        video_ids.append(item['contentDetails']['videoId'])

    # Check if there are more pages of results
    next_page_token = response.get('nextPageToken')

    # Loop to keep fetching video IDs from the next pages
    while next_page_token is not None:
        request = youtube.playlistItems().list(
            part="snippet, contentDetails",
            playlistId=playlist_id,
            maxResults=50,
            pageToken=next_page_token  # Use the token for the next page
        )
        response = request.execute()

        # Extract video IDs from the current page
        for item in response['items']:
            video_ids.append(item['contentDetails']['videoId'])

        # Update the token for the next iteration
        next_page_token = response.get('nextPageToken')

    return video_ids

In [ ]:
# Get video IDs
video_ids = get_video_ids(youtube, playlist_id)

In [ ]:
len(video_ids)

In [ ]:
video_ids

In [ ]:

def get_video_details(youtube, video_ids):
    all_video_info = []

    # Loop through video_ids in chunks of 50
    for i in range(0, len(video_ids), 50):
        request = youtube.videos().list(
            part="snippet,statistics,contentDetails,status,recordingDetails,topicDetails",
            id=','.join(video_ids[i:i + 50])  # Join IDs in the current chunk
        )
        response = request.execute()

        # Process each video in the response
        for video in response.get('items', []):
            stats_to_keep = {
                'snippet': [
                    'channelTitle', 'title', 'description', 'tags', 'publishedAt',
                    'categoryId', 'channelId', 'defaultLanguage', 'defaultAudioLanguage'
                ],
                'statistics': [
                    'viewCount', 'likeCount', 'favoriteCount', 'commentCount', 'dislikeCount'
                ],
                'contentDetails': [
                    'duration', 'definition', 'caption',
                ],
                'status': [
                    'madeForKids'
                ],
                'recordingDetails': [
                    'recordingDate', 'location'
                ],
                'topicDetails': [
                    'topicCategories'
                ],
            }

            video_info = {}
            video_info['video_id'] = video.get('id')
            # Add the URL of the video
            video_info['video_url'] = f"https://www.youtube.com/watch?v={video_info['video_id']}"

            # Loop through the categories of information we want to keep
            for k in stats_to_keep.keys():
                data = video.get(k, {})
                for v in stats_to_keep[k]:
                    value = data.get(v)
                    # Handle nested dictionaries
                    if isinstance(value, dict):
                        video_info[v] = json.dumps(value)
                    else:
                        video_info[v] = value

            all_video_info.append(video_info)

    # Return DataFrame containing all video information
    return pd.DataFrame(all_video_info)

In [ ]:
all_video_info = get_video_details(youtube, video_ids)
all_video_info

In [ ]:
all_video_info.columns

In [ ]:
# Path to the shared unit in Google Drive
base_folder = '/content/gdrive/MyDrive/David Bisbal/'

# Make sure that the path exits in the repository
if not os.path.exists(base_folder):
    os.makedirs(base_folder)

    # Assuming you have the channel name in `channelTitle`.
for channel_title in all_video_info['channelTitle']:
    # Create a folder named after the channel inside the base directory.
    channel_folder = os.path.join(base_folder, channel_title)

    # Check whether the channel folder already exists; if not, create it.
    if not os.path.exists(channel_folder):
        os.makedirs(channel_folder)

    # Create the CSV file name.
    filename = f"ChStat_{channel_title}.csv"

    # Save the CSV file for that channel only.
    all_video_info.to_csv(filename, index=False, sep='\t')

    # Move the CSV file to the channel folder.
    destination = os.path.join(channel_folder, filename)
    !cp -r "/content/{filename}" "{destination}"

# VIDEO COMMENTS 

In [ ]:
def get_comments_in_videos(youtube, video_ids):
    all_comments = []

    for video_id in video_ids:
        next_page_token = None  # Initialize the pagination token.

        while True:
            try:
                request = youtube.commentThreads().list(
                    part="snippet,replies",
                    videoId=video_id,
                    pageToken=next_page_token,
                    maxResults=100  # Set `maxResults` to retrieve more comments at once.
                )
                response = request.execute()

                # Extract the comments from the response.
                for item in response.get('items', []):
                    comment = item['snippet']['topLevelComment']
                    snippet = comment['snippet']

                    comment_data = {
                        'video_id': video_id,
                        'comment_id': comment['id'],
                        'author_name': snippet.get('authorDisplayName'),
                        'author_channel_id': snippet.get('authorChannelId', {}).get('value'),
                        'comment_text': snippet.get('textOriginal'),
                        'published_at': snippet.get('publishedAt'),
                        'updated_at': snippet.get('updatedAt'),
                        'like_count': snippet.get('likeCount'),
                        'total_reply_count': item['snippet'].get('totalReplyCount'),
                        'can_reply': item['snippet'].get('canReply'),
                        'is_public': snippet.get('isPublic'),
                        'viewer_rating': snippet.get('viewerRating'),
                        'moderation_status': snippet.get('moderationStatus'),
                        'author_profile_image_url': snippet.get('authorProfileImageUrl'),
                    }

                    all_comments.append(comment_data)

                # Verification of pagination
                next_page_token = response.get('nextPageToken')
                if not next_page_token:  # Exit the loop if there are no more pages.
                    break

            except HttpError as e:
                if e.resp.status == 403:
                    print(f"Comments are disabled for video ID {video_id}.")
                else:
                    print(f"An error occurred for video ID {video_id}: {e}")
                break  # Exit the `while` loop for this video in case of an error.

    # Create a DataFrame with all metadatas
    return pd.DataFrame(all_comments)

# Assume `youtube` and `video_ids` are defined
comment_df = get_comments_in_videos(youtube, video_ids)

In [ ]:
comment_df

In [ ]:
# Google Drive shared drive path
base_folder = '/content/gdrive/MyDrive/David Bisbal/'

# Ensure the destination folder's base path exists
if not os.path.exists(base_folder):
    os.makedirs(base_folder)

# Assuming you have the channel name in `channelTitle`
for channel_title in all_video_info['channelTitle']:
    # Create the folder named after the channel within the base path
    channel_folder = os.path.join(base_folder, channel_title)

    # Check whether the channel folder already exists; if not, create it
    if not os.path.exists(channel_folder):
        os.makedirs(channel_folder)

    # Create the CSV name file
    filename1 = f"ComStat_{channel_title}.csv"

    # Save the CSV file for that channel only.
    comment_df.to_csv(filename1, index=False, sep='\t', escapechar='\\')

    # Move the CSV file to the channel folder.
    destination = os.path.join(channel_folder, filename1)
    !cp -r "/content/{filename1}" "{destination}"

# DOWNLOADING THE VIDEOS

In [ ]:
# Assuming you have a DataFrame containing the video URLs and `channelTitle`
video_details_df = get_video_details(youtube, video_ids)

# Extract the video URLs from the DataFrame
video_urls = video_details_df['video_url'].tolist()

# Download each video and save it in the `media` folder within the corresponding channel folder
for url in video_urls:
    # Get the video's `channelTitle` (make sure this column exists in the DataFrame)
    channel_title = video_details_df.loc[video_details_df['video_url'] == url, 'channelTitle'].values[0]

    # Create the path to the channel folder and its `media` subfolder
    channel_folder = os.path.join(base_folder, channel_title)
    media_folder = os.path.join(channel_folder, 'media')

    # Check whether the `media` folder exists; if not, create it
    if not os.path.exists(media_folder):
        os.makedirs(media_folder)

    # Set the output file path for the downloaded video
    output_file = os.path.join(media_folder, "%(title)s.%(ext)s")

    # Download the video using `yt-dlp` and save it in the channel's `media` folder
    subprocess.run([
        "yt-dlp",
        "-f", "134+bestaudio",  # ID para 480p
        "--merge-output-format", "mp4",
        "-o", output_file,  # Guardar en la carpeta 'media' del canal
        url
    ])

Now, as we have data needing, we can pass to the analysis part.